# `c01_ic` — Institutional Characteristics and the Directory Universe

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `HD2023`, `IC2023`, `IC2023_AY`, `DRVIC2023` |
| Reference period | 2023-24 institutional universe; AY 2023-24 published charges |
| Curated grain | `UNITID` |
| Output | `data/curated/c01_ic.parquet` |

This notebook produces the peer-group spine every other component joins against. Filter to `CYACTIVE == 1` for analyses that require a currently active institution, but keep inactive rows in the curated table so that closures remain visible to the longitudinal notebooks rather than silently vanishing from a panel.

> **Pitfall.** Negative values are not data. IPEDS uses -1, -2, and -3 as reserved missing codes; a mean computed without masking them is badly wrong and looks fine.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c01_ic"
TABLES = ['HD2023', 'IC2023', 'IC2023_AY', 'DRVIC2023']
GRAIN = ['UNITID']
REFERENCE_PERIOD = '2023-24 institutional universe; AY 2023-24 published charges'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,HD2023,1110720,e11d35af6f50fbe2f51d8ddd5a9d4f49860abbab7d73be...,2026-09-24T17:19:09+00:00
1,IC2023,381806,1454ab8fc5df34aafb04566351c29ed313350f41bc2a24...,2026-09-24T17:19:09+00:00
2,IC2023_AY,309736,42d3ee39a107d69b6da02df2ffa934ebe3bb76657568fb...,2026-09-24T17:19:09+00:00
3,DRVIC2023,73481,7fd4c8671bec2276ed3272f118c468316316e81d971fd5...,2026-09-24T17:19:09+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'2023-24',
    table=TABLES[0],
)
print(intro[:600])

File Documentation for the IPEDS Directory, 2023-24
(Release 4)
Filename HD2023
Release 1: August 2024
Release 2: September 2024
Release 3: January 2025
Release 4: September 2025
Overview This file contains directory information for every institution in the 2023-24 IPEDS universe.  Includes name, address, city, state, zip code and various URL links to the institution's home page, admissions, financial aid offices and  the net price calculator.  Identifies institutions as currently active, and institutions that participate in Title IV federal financial aid programs for which IPEDS is mandatory.


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

73 variables documented, 4031 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,INSTNM,Institution (entity) name
2,IALIAS,Institution name alias
3,ADDR,Street address or post office box
4,CITY,City location of institution
5,STABBR,State abbreviation
6,ZIP,ZIP code
7,FIPS,FIPS state code
8,OBEREG,Bureau of Economic Analysis (BEA) regions
9,CHFNM,Name of chief administrator


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'INSTNM', 'STABBR', 'FIPS', 'OBEREG', 'SECTOR', 'ICLEVEL', 'CONTROL', 'HLOFFER', 'UGOFFER', 'GROFFER', 'HBCU', 'TRIBAL', 'LOCALE', 'C21BASIC', 'CYACTIVE', 'INSTCAT']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (6163, 73)
schema: unchanged | added [] | removed []


,UNITID,INSTNM,STABBR,FIPS,OBEREG,SECTOR,ICLEVEL,CONTROL,HLOFFER,UGOFFER,GROFFER,HBCU,TRIBAL,LOCALE,C21BASIC,CYACTIVE,INSTCAT
0,100654,Alabama A & M University,AL,1,5,1,1,1,9,1,1,1,2,12,18,1,2
1,100663,University of Alabama at Birmingham,AL,1,5,1,1,1,9,1,1,2,2,12,15,1,2
2,100690,Amridge University,AL,1,5,2,1,2,9,1,1,2,2,12,20,1,2
3,100706,University of Alabama in Huntsville,AL,1,5,1,1,1,9,1,1,2,2,12,15,1,2
4,100724,Alabama State University,AL,1,5,1,1,1,9,1,1,1,2,12,17,1,2


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in ['CONTROL', 'SECTOR', 'ICLEVEL', 'LOCALE', 'C21BASIC']:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 2,643 reserved-code cells across 14 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

No X-prefixed imputation flags accompany this file.


## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = ['CONTROL', 'SECTOR', 'ICLEVEL', 'LOCALE', 'C21BASIC']

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,CONTROL,SECTOR,ICLEVEL,LOCALE,C21BASIC,CONTROL_LABEL,SECTOR_LABEL,ICLEVEL_LABEL,LOCALE_LABEL,C21BASIC_LABEL
0,1,1,1,12,18,Public,"Public, 4-year or above",Four or more years,City: Midsize,Master's Colleges & Universities: Larger Programs
1,1,1,1,12,15,Public,"Public, 4-year or above",Four or more years,City: Midsize,Doctoral Universities: Highest Research Activity
2,2,2,1,12,20,Private not-for-profit,"Private not-for-profit, 4-year or above",Four or more years,City: Midsize,Master's Colleges & Universities: Small Programs
4,1,1,1,12,17,Public,"Public, 4-year or above",Four or more years,City: Midsize,Doctoral/Professional Universities
5,1,0,1,12,<NA>,Public,Administrative Unit,Four or more years,City: Midsize,NaN
7,1,4,2,32,5,Public,"Public, 2-year",At least 2 but less than 4 years,Town: Distant,Associate's Colleges: Mixed Transfer/Career & ...
8,1,1,1,31,22,Public,"Public, 4-year or above",Four or more years,Town: Fringe,Baccalaureate Colleges: Diverse Fields
10,1,1,1,13,15,Public,"Public, 4-year or above",Four or more years,City: Small,Doctoral Universities: Highest Research Activity
11,2,2,1,12,21,Private not-for-profit,"Private not-for-profit, 4-year or above",Four or more years,City: Midsize,Baccalaureate Colleges: Arts & Sciences Focus
12,1,4,2,21,4,Public,"Public, 2-year",At least 2 but less than 4 years,Suburb: Large,Associate's Colleges: Mixed Transfer/Career & ...


## 9. Reshape to the declared grain

Target grain: `UNITID`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID'] -> 6,163 rows, 0 duplicated


,UNITID,INSTNM,STABBR,FIPS,OBEREG,SECTOR,ICLEVEL,CONTROL,HLOFFER,UGOFFER,GROFFER,HBCU,TRIBAL,LOCALE,C21BASIC,CYACTIVE,INSTCAT,CONTROL_LABEL,SECTOR_LABEL,ICLEVEL_LABEL,LOCALE_LABEL,C21BASIC_LABEL
0,100654,Alabama A & M University,AL,1.0,5.0,1,1,1,9.0,1.0,1.0,1.0,2.0,12,18,1.0,2.0,Public,"Public, 4-year or above",Four or more years,City: Midsize,Master's Colleges & Universities: Larger Programs
1,100663,University of Alabama at Birmingham,AL,1.0,5.0,1,1,1,9.0,1.0,1.0,2.0,2.0,12,15,1.0,2.0,Public,"Public, 4-year or above",Four or more years,City: Midsize,Doctoral Universities: Highest Research Activity
2,100690,Amridge University,AL,1.0,5.0,2,1,2,9.0,1.0,1.0,2.0,2.0,12,20,1.0,2.0,Private not-for-profit,"Private not-for-profit, 4-year or above",Four or more years,City: Midsize,Master's Colleges & Universities: Small Programs
3,100706,University of Alabama in Huntsville,AL,1.0,5.0,1,1,1,9.0,1.0,1.0,2.0,2.0,12,15,1.0,2.0,Public,"Public, 4-year or above",Four or more years,City: Midsize,Doctoral Universities: Highest Research Activity
4,100724,Alabama State University,AL,1.0,5.0,1,1,1,9.0,1.0,1.0,1.0,2.0,12,17,1.0,2.0,Public,"Public, 4-year or above",Four or more years,City: Midsize,Doctoral/Professional Universities


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID'),
    iu.not_null('UNITID', 'INSTNM'),
    iu.subset_of('CONTROL', {1, 2, 3, -3}),
    iu.in_range('SECTOR', 0, 99),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,unique_key(UNITID),pass,0,0.0,Declared grain must be unique
1,"not_null(UNITID,INSTNM)",pass,0,0.0,Key columns must be populated
2,subset_of(CONTROL),pass,0,0.0,Codes must match published value set
3,"in_range(SECTOR,0,99)",pass,0,0.0,Value plausibility bound


PASSED


Report(table='c01_ic', rows=6163, results=[{'name': 'unique_key(UNITID)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'not_null(UNITID,INSTNM)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Key columns must be populated', 'status': 'pass'}, {'name': 'subset_of(CONTROL)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Codes must match published value set', 'status': 'pass'}, {'name': 'in_range(SECTOR,0,99)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}], generated_utc='2026-09-24T17:19:10+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes='Negative values are not data. IPEDS uses -1, -2, and -3 as reserved missing codes; a mean computed without masking them is badly wrong and looks fine.',
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c01_ic.parquet (6,163 rows x 22 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. Negative values are not data. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.